# ❄️ Conjugate Heat Transfer (CHT) with Extended Physics-Informed Neural Networks (XPINNs)
### Modeling Coupled Fluid Convection, Solid Conduction, and Interface Slope Discontinuities
**Author:** Mohammed Basra (`mohammedbasra-star`)  

This tutorial demonstrates:
1. **Conjugate Heat Transfer Coupling:** Solid conduction coupled with fluid convective laminar channel flow.
2. **Why Monolithic PINNs Fail:** Single networks cannot capture the steep slope jump in normal temperature gradient ($k_s/k_f = 10$).
3. **XPINN Domain Decomposition:** Dual-subnet architecture communicating via temperature and heat flux transmission conditions.

In [ ]:
import os, sys
import torch
import numpy as np
import matplotlib.pyplot as plt

# Add repository root
sys.path.append(os.path.abspath('..'))

from physics.analytical_benchmarks import solve_conjugate_ground_truth
from models.xpinn_module import XPINNConjugate
from models.monolithic_pinn import MonolithicPINN

## 1. High-Precision Finite Volume Ground Truth Benchmark

In [ ]:
gt = solve_conjugate_ground_truth(nx=60, ny_solid=30, ny_fluid=30, k_ratio=10.0, pe=10.0)

fig, ax = plt.subplots(figsize=(10, 4.5))
X_s, Y_s = np.meshgrid(gt['x'], gt['y_solid'], indexing='ij')
X_f, Y_f = np.meshgrid(gt['x'], gt['y_fluid'], indexing='ij')

ax.contourf(X_s, Y_s, gt['T_solid'], levels=40, cmap='hot')
c = ax.contourf(X_f, Y_f, gt['T_fluid'], levels=40, cmap='hot')
ax.axhline(0.0, color='cyan', linestyle='--', lw=2, label='Solid-Fluid Interface y=0')
ax.set_xlabel('Axial Position x (m)')
ax.set_ylabel('Transverse Position y (m)')
ax.set_title('Conjugate Heat Transfer Ground Truth Temperature Field')
plt.colorbar(c, ax=ax, label='Temperature T')
plt.legend()
plt.tight_layout()
plt.show()

## 2. Real-Time Inference with Trained XPINN Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
xpinn = XPINNConjugate(hidden_dim=64, num_layers=4).to(device)

checkpoint_path = '../results/best_xpinn_model.pt'
if os.path.exists(checkpoint_path):
    xpinn.load_state_dict(torch.load(checkpoint_path, map_location=device))
    xpinn.eval()
    print('Loaded pretrained XPINN weights successfully!')
else:
    print('Initialized model (run train.py to train complete weights)')

# Evaluate vertical line cut across interface
x_val = 1.0
x_tensor = torch.full((50, 1), x_val, dtype=torch.float32).to(device)
y_s_tensor = torch.linspace(-0.5, 0.0, 50).unsqueeze(1).to(device)
y_f_tensor = torch.linspace(0.0, 0.5, 50).unsqueeze(1).to(device)

with torch.no_grad():
    T_s_line = xpinn.forward_solid(x_tensor, y_s_tensor).cpu().numpy().ravel()
    T_f_line = xpinn.forward_fluid(x_tensor, y_f_tensor).cpu().numpy().ravel()

y_comb = np.concatenate([y_s_tensor.cpu().numpy().ravel(), y_f_tensor.cpu().numpy().ravel()])
T_comb = np.concatenate([T_s_line, T_f_line])

plt.figure(figsize=(7, 4.5))
plt.plot(y_comb, T_comb, 'r-', lw=2, label='XPINN Temperature')
plt.axvline(0.0, color='gray', linestyle='--', label='Interface (y = 0)')
plt.xlabel('Transverse Height y (m)')
plt.ylabel('Temperature T')
plt.title(f'XPINN Interface Slope Discontinuity at x = {x_val} m')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()